In [11]:
import pandas as pd

df = pd.read_csv("customer_support_text_classification.csv")

print(df.head())

print("\nShape:")
print(df.shape)

print("\nColumns:")
print(df.columns)

print("\nClass Distribution:")
print(df['sentiment_label'].value_counts())

print("\nAverage Text Length:")
print(df['customer_message'].apply(len).mean())

  ticket_id channel                                   customer_message  \
0  TKT00001    chat  I need information about the payment process. ...   
1  TKT00002   phone      I need information about the payment process.   
2  TKT00003   email  The refund process was fast and convenient. I ...   
3  TKT00004  social  My refund is still pending and this experience...   
4  TKT00005    chat   Please tell me how to update my account details.   

  sentiment_label  word_count  urgent_flag  
0         neutral          18            1  
1         neutral           7            0  
2        positive          12            0  
3        negative          15            1  
4         neutral           9            0  

Shape:
(1500, 6)

Columns:
Index(['ticket_id', 'channel', 'customer_message', 'sentiment_label',
       'word_count', 'urgent_flag'],
      dtype='str')

Class Distribution:
sentiment_label
neutral     524
negative    497
positive    479
Name: count, dtype: int64

Average Text Length

In [12]:
print("\nSample Text Records:\n")

for i in range(5):
    print(df['customer_message'][i])
    print()


Sample Text Records:

I need information about the payment process. My ticket number is 78732. Please respond as soon as possible.

I need information about the payment process.

The refund process was fast and convenient. I appreciate the quick response.

My refund is still pending and this experience is frustrating. My ticket number is 33927.

Please tell me how to update my account details.



In [13]:
import re
import nltk

from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('punkt')
nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

def preprocess_text(text):

    text = text.lower()

    text = re.sub(r'[^a-zA-Z\s]', '', text)

    tokens = word_tokenize(text)

    tokens = [word for word in tokens if word not in stop_words]

    return " ".join(tokens)

df['cleaned_text'] = df['customer_message'].apply(preprocess_text)

print(df[['customer_message', 'cleaned_text']].head())

                                    customer_message  \
0  I need information about the payment process. ...   
1      I need information about the payment process.   
2  The refund process was fast and convenient. I ...   
3  My refund is still pending and this experience...   
4   Please tell me how to update my account details.   

                                        cleaned_text  
0  need information payment process ticket number...  
1                   need information payment process  
2  refund process fast convenient appreciate quic...  
3  refund still pending experience frustrating ti...  
4                 please tell update account details  


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\yashd\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\yashd\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [14]:
from sklearn.feature_extraction.text import TfidfVectorizer

vectorizer = TfidfVectorizer(max_features=5000)

X = vectorizer.fit_transform(df['cleaned_text'])

y = df['sentiment_label']

print(X.shape)

(1500, 146)


In [15]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import pandas as pd

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

print("\nClassification Report:")
print(classification_report(y_test, predictions))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, predictions))

Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       109
     neutral       1.00      1.00      1.00       104
    positive       1.00      1.00      1.00        87

    accuracy                           1.00       300
   macro avg       1.00      1.00      1.00       300
weighted avg       1.00      1.00      1.00       300


Confusion Matrix:
[[109   0   0]
 [  0 104   0]
 [  0   0  87]]


In [16]:
results_df = pd.DataFrame(classification_report(
    y_test,
    predictions,
    output_dict=True
)).transpose()

results_df.to_csv("results/model_evaluation.csv")

print(results_df)

              precision  recall  f1-score  support
negative            1.0     1.0       1.0    109.0
neutral             1.0     1.0       1.0    104.0
positive            1.0     1.0       1.0     87.0
accuracy            1.0     1.0       1.0      1.0
macro avg           1.0     1.0       1.0    300.0
weighted avg        1.0     1.0       1.0    300.0


In [20]:
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Dense, Dropout
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

tokenizer = Tokenizer(num_words=5000)

tokenizer.fit_on_texts(df['cleaned_text'])

sequences = tokenizer.texts_to_sequences(df['cleaned_text'])

max_length = 50

X_seq = pad_sequences(sequences, maxlen=max_length)

label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(df['sentiment_label'])

X_train_seq, X_test_seq, y_train_seq, y_test_seq = train_test_split(
    X_seq,
    y_encoded,
    test_size=0.2,
    random_state=42
)

lstm_model = Sequential([
    Input(shape=(50,)),
    Embedding(input_dim=5000, output_dim=64),
    LSTM(64),
    Dropout(0.5),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')
])

lstm_model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

lstm_model.summary()

history = lstm_model.fit(
    X_train_seq,
    y_train_seq,
    validation_data=(X_test_seq, y_test_seq),
    epochs=5,
    batch_size=32
)

lstm_predictions = lstm_model.predict(X_test_seq)

lstm_predicted_classes = lstm_predictions.argmax(axis=1)

print("LSTM Accuracy:", accuracy_score(y_test_seq, lstm_predicted_classes))

print("\nClassification Report:")
print(classification_report(
    y_test_seq,
    lstm_predicted_classes,
    target_names=label_encoder.classes_
))

sample_texts = [
    "The refund process was fast and convenient.",
    "My refund is still pending and this experience is frustrating.",
    "Please tell me how to update my account details.",
    "I am very happy with the customer support service.",
    "The issue has not been resolved yet."
]

sample_sequences = tokenizer.texts_to_sequences(sample_texts)

sample_padded = pad_sequences(sample_sequences, maxlen=50)

predictions = lstm_model.predict(sample_padded)

predicted_classes = predictions.argmax(axis=1)

predicted_labels = label_encoder.inverse_transform(predicted_classes)

with open("results/sample_predictions.txt", "w") as file:

    for text, label in zip(sample_texts, predicted_labels):

        file.write(f"Input: {text}\n")
        file.write(f"Prediction: {label}\n\n")

print("sample_predictions.txt created successfully")

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ (None, 50, 64)         │       320,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_5 (LSTM)                   │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_5 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_10 (Dense)                │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_11 (Dense)                │ (None, 3)              │            99 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 355,203 (1.35 MB)

 Trainable params: 355,203 (1.35 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 2s 17ms/step - accuracy: 0.6083 - loss: 1.0452 - val_accuracy: 0.8333 - val_loss: 0.8337
Epoch 2/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.9400 - loss: 0.3561 - val_accuracy: 1.0000 - val_loss: 0.0200
Epoch 3/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 13ms/step - accuracy: 1.0000 - loss: 0.0106 - val_accuracy: 1.0000 - val_loss: 5.9138e-04
Epoch 4/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 1.0000 - loss: 0.0029 - val_accuracy: 1.0000 - val_loss: 2.5481e-04
Epoch 5/5
38/38 ━━━━━━━━━━━━━━━━━━━━ 1s 14ms/step - accuracy: 1.0000 - loss: 0.0019 - val_accuracy: 1.0000 - val_loss: 1.5515e-04
10/10 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step
LSTM Accuracy: 1.0

Classification Report:
              precision    recall  f1-score   support

    negative       1.00      1.00      1.00       109
     neutral       1.00      1.00      1.00       104
    positive       1.00      1.00      1.00        87

    accuracy                           1.00       300
   

# Task 1: Dataset Understanding

The data was imported and processed in Pandas.

There are 6 columns and 1500 records in the dataset.

The class to be targeted is sentiment_label, with 3 classes:
- positive
- neutral
- negative

The primary text feature that NLP handles for is NLP processing is `customer_message`.

A number of sample text records were presented to gain insight into the structure of customer support text records.

The mean length of the text was around 72 characters.

A class distribution was created to see if the collection was relatively balanced within the three sentiment classes.

---

# Task 2: Text Preprocessing

Text data has been preprocessed and cleaned prior to model training.

The following pre-processing steps were taken:
- Lowering a text to lowercase
- removing special characters and symbols
- tokenizing the text
- removing stopwords

The cleaned text was then saved in a new column called `cleaned_text`.

Preprocessing the text data can help remove noise from the data and enhance the quality of the features fed into the machine learning models.

---

# Task 3: Text Vectorization

The cleaned text was then vectorised into a numerical format, using TF-IDF.

TF-IDF (Term Frequency-Inverse Document Frequency) is used to assess the significance of words in text documents, and to minimize the effect of other frequent words in the documents.

The dataset used in this analysis was vectorized (1500 rows, 146 numerical features).

Machine learning models cannot work with raw text data; therefore, it is necessary to convert the text into vectors. To train and predict models, numerical representations are needed.

---

# Task 4: Baseline Model

Logistic Regression model was used as a baseline model along with TF-IDF vectorization.

A 80/20 split was used in the training and testing sets of the data.

The model was very accurate and classified customers' sentiment as positive, neutral, and negative.

Evaluation metrics included:
- accuracy
- precision
- recall
- F1-score
- confusion matrix

The classification report indicated that all the sentiment classes performed well.

---

# Task 5: Sequence Model

The TensorFlow/Keras based LSTM (Long Short-Term Memory) sequence model was developed.

The following model architecture was included:
- input sequence processing
- embedding layer
- LSTM layer
- dropout layer
- dense output layer

The embedding layer was what converted words to dense vector representations.

The sequential information of the text and the context relation between each word were captured and learned in the LSTM layer.

The output layer had softmax activation and classified text into three sentiment classes.

The model used:
We will use sparse categorical cross-entropy as the loss function:
- using accuracy as the metric for evaluation

The performance for the testing set was good with the sequence model.

---
# Task 6: Attention and Transformer Reflection

The problem with RNNs is that they find it difficult to capture long-range dependencies because information from the past can be forgotten over time.

LSTMs use gates to determine what information to remember and what information to forget, which makes them better at storing memories.

Attention mechanisms enable models to attend to relevant words in a sequence rather than all words.

Transformers play a crucial role in modern NLP and Generative AI due to their efficient processing of entire sequences and their ability to perform well on complex language tasks.